In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from sklearn_crfsuite import CRF
from sklearn_crfsuite.metrics import flat_classification_report, flat_f1_score
from sklearn.metrics import accuracy_score

dataset_path = 'C:/Users/lars/Documents/School/group50-text-mining/Poster project/data/NER dataset.csv'
test_path = 'C:/Users/lars/Documents/School/group50-text-mining/Poster project/test_sets/NER-test.tsv'

def read_table_with_fallback(filepath, **kwargs):
    """Read a delimited text file using a small set of common encodings."""
    encodings = kwargs.pop('encodings', ['utf-8', 'utf-8-sig', 'cp1252', 'latin1'])
    last_error = None
    for encoding in encodings:
        try:
            return pd.read_csv(filepath, encoding=encoding, **kwargs)
        except UnicodeDecodeError as exc:
            last_error = exc
    raise last_error

def parse_ner_csv(filepath):
    """
    Parse NER dataset CSV format.
    Returns list of sentences, where each sentence is a list of token dictionaries.
    """
    df = read_table_with_fallback(filepath)
    sentences = []
    current_sentence = []
    current_sent_id = None

    for idx, row in df.iterrows():
        sent_id = str(row['Sentence #']).split(':')[0].strip()

        # Check if we're on a new sentence
        if sent_id != current_sent_id:
            if current_sentence:
                sentences.append(current_sentence)
            current_sentence = []
            current_sent_id = sent_id

        current_sentence.append({
            'word': row['Word'],
            'pos': row['POS'],
            'ner': row['Tag']
        })

    # Don't forget the last sentence
    if current_sentence:
        sentences.append(current_sentence)

    return sentences

def parse_ner_tsv(filepath):
    """
    Parse NER-test.tsv format.
    Returns list of sentences, where each sentence is a list of token dictionaries.
    """
    df = read_table_with_fallback(filepath, sep='\t')
    sentences = []
    current_sentence = []
    current_sent_id = None

    for idx, row in df.iterrows():
        sent_id = row['sentence id']

        # Check if we're on a new sentence
        if sent_id != current_sent_id:
            if current_sentence:
                sentences.append(current_sentence)
            current_sentence = []
            current_sent_id = sent_id

        current_sentence.append({
            'word': row['token'],
            'pos': '',
            'ner': row['BIO NER tag']
        })

    if current_sentence:
        sentences.append(current_sentence)

    return sentences

train_sentences = parse_ner_csv(dataset_path)
test_sentences = parse_ner_tsv(test_path)

print(f"Train sentences: {len(train_sentences)}")
print(f"Test sentences: {len(test_sentences)}")


Train sentences: 95914
Test sentences: 10


In [28]:
# Normalize training/test labels to common CoNLL-style space
# Keep BIO prefix, map entity type part.
ENTITY_MAP = {
    "geo": "LOC",
    "gpe": "LOC",
    "loc": "LOC",
    "org": "ORG",
    "per": "PER",
    "art": "MISC",
    "eve": "MISC",
    "nat": "MISC",
    "tim": "MISC",
    "misc": "MISC",
    "LOC": "LOC",
    "ORG": "ORG",
    "PER": "PER",
    "MISC": "MISC",
}

def normalize_tag(tag):
    if not isinstance(tag, str):
        return "O"
    tag = tag.strip()
    if tag == "O" or tag == "":
        return "O"
    if "-" not in tag:
        return "O"
    bio, ent = tag.split("-", 1)
    ent_norm = ENTITY_MAP.get(ent, ENTITY_MAP.get(ent.lower(), "MISC"))
    bio = "B" if bio.upper().startswith("B") else "I"
    return f"{bio}-{ent_norm}"

def normalize_sentences_labels(sentences):
    for sent in sentences:
        for tok in sent:
            tok["ner"] = normalize_tag(tok.get("ner", "O"))
    return sentences

train_sentences = normalize_sentences_labels(train_sentences)
test_sentences = normalize_sentences_labels(test_sentences)

# Quick check
train_label_set = sorted({tok["ner"] for s in train_sentences for tok in s})
test_label_set = sorted({tok["ner"] for s in test_sentences for tok in s})
print("Train labels:", train_label_set)
print("Test labels:", test_label_set)

Train labels: ['B-LOC', 'B-MISC', 'B-ORG', 'B-PER', 'I-LOC', 'I-MISC', 'I-ORG', 'I-PER', 'O']
Test labels: ['B-LOC', 'B-MISC', 'B-ORG', 'B-PER', 'I-LOC', 'I-MISC', 'I-ORG', 'I-PER', 'O']


In [29]:

def extract_ner_tags(sentences):
    """Extract all NER tags from sentences."""
    tags = []
    for sentence in sentences:
        for token in sentence:
            tags.append(token['ner'])
    return tags

def extract_ner_entities(sentences):
    """
    Extract entities (not just tags).
    Returns list of (entity_type, entity_text) tuples.
    """
    entities = []
    for sentence in sentences:
        current_type = None
        current_words = []

        for token in sentence:
            tag = token['ner']
            word = token['word']

            if tag == 'O':
                # End current entity if exists
                if current_words:
                    entity_text = ' '.join(current_words)
                    entities.append((current_type, entity_text))
                    current_words = []
                    current_type = None
            else:
                # Parse BIO tag (e.g., 'B-PER', 'I-PER')
                if '-' in tag:
                    bio, ent_type = tag.split('-', 1)
                else:
                    bio, ent_type = tag, tag

                if bio == 'B' or (current_type != ent_type and current_words):
                    # Start new entity
                    if current_words:
                        entity_text = ' '.join(current_words)
                        entities.append((current_type, entity_text))
                    current_words = [word]
                    current_type = ent_type
                else:
                    # Continue entity
                    current_words.append(word)
                    current_type = ent_type

        # Don't forget the last entity in sentence
        if current_words:
            entity_text = ' '.join(current_words)
            entities.append((current_type, entity_text))

    return entities

def get_entity_types(entities):
    """Extract entity types from entities list."""
    return [ent_type for ent_type, _ in entities]

# Extract data for analysis
train_tags = extract_ner_tags(train_sentences)
test_tags = extract_ner_tags(test_sentences)

train_entities = extract_ner_entities(train_sentences)
test_entities = extract_ner_entities(test_sentences)

In [30]:
def count_instances(sentences, tags, entities):
    """Count various instances in the dataset."""
    return {
        'num_sentences': len(sentences),
        'num_tokens': len(tags),
        'num_entities': len(entities),
        'unique_entity_types': len(set(get_entity_types(entities))),
        'unique_tags': len(set(tags))
    }

train_counts = count_instances(train_sentences, train_tags, train_entities)
test_counts = count_instances(test_sentences, test_tags, test_entities)

# Create summary dataframe
counts_df = pd.DataFrame({
    'Train': train_counts,
    'Test': test_counts
})
print(counts_df)


                       Train  Test
num_sentences          95914    10
num_tokens           1048575   214
num_entities          115929    17
unique_entity_types        4     4
unique_tags                9     9


In [31]:
# Tag frequency distributions
train_tag_freq = Counter(train_tags)
test_tag_freq = Counter(test_tags)

# Create comparison dataframe
all_tags = set(train_tag_freq.keys()) | set(test_tag_freq.keys())
tag_comparison = pd.DataFrame({
    'Train': [train_tag_freq[tag] for tag in sorted(all_tags)],
    'Test': [test_tag_freq[tag] for tag in sorted(all_tags)]
}, index=sorted(all_tags))

# Calculate percentages
tag_percentages = tag_comparison.div(tag_comparison.sum(axis=0), axis=1) * 100

print("NER TAG DISTRIBUTION (Counts)")
print(tag_comparison, '\n')
print("NER TAG DISTRIBUTION (Percentages)")
print(tag_percentages.round(2))

NER TAG DISTRIBUTION (Counts)
         Train  Test
B-LOC    53514     4
B-MISC   21244     3
B-ORG    20143     4
B-PER    16990     6
I-LOC     7612     2
I-MISC    7129     1
I-ORG    16784     3
I-PER    17251     8
O       887908   183 

NER TAG DISTRIBUTION (Percentages)
        Train   Test
B-LOC    5.10   1.87
B-MISC   2.03   1.40
B-ORG    1.92   1.87
B-PER    1.62   2.80
I-LOC    0.73   0.93
I-MISC   0.68   0.47
I-ORG    1.60   1.40
I-PER    1.65   3.74
O       84.68  85.51


In [32]:
# Entity type frequencies (excluding 'O' tag)
train_entity_types = [ent_type for ent_type, _ in train_entities]
test_entity_types = [ent_type for ent_type, _ in test_entities]

train_ent_freq = Counter(train_entity_types)
test_ent_freq = Counter(test_entity_types)

# Create comparison dataframe
all_ent_types = set(train_ent_freq.keys()) | set(test_ent_freq.keys())
entity_comparison = pd.DataFrame({
    'Train': [train_ent_freq[ent_type] for ent_type in sorted(all_ent_types)],
    'Test': [test_ent_freq[ent_type] for ent_type in sorted(all_ent_types)]
}, index=sorted(all_ent_types))

# Calculate percentages
entity_percentages = entity_comparison.div(entity_comparison.sum(axis=0), axis=1) * 100

print("ENTITY TYPE DISTRIBUTION (Counts)")
print(entity_comparison, '\n')
print("ENTITY TYPE DISTRIBUTION (Percentages)")
print(entity_percentages.round(2))

ENTITY TYPE DISTRIBUTION (Counts)
      Train  Test
LOC   53832     4
MISC  21328     3
ORG   21031     4
PER   19738     6 

ENTITY TYPE DISTRIBUTION (Percentages)
      Train   Test
LOC   46.44  23.53
MISC  18.40  17.65
ORG   18.14  23.53
PER   17.03  35.29


In [33]:
def sent2features(sent):
    features = []
    for i, token in enumerate(sent):
        word = '' if pd.isna(token.get('word', '')) else str(token.get('word', ''))
        pos = '' if pd.isna(token.get('pos', '')) else str(token.get('pos', ''))
        feat = {
            'bias': 1.0,
            'word.lower()': word.lower(),
            'word[-3:]': word[-3:],
            'word[-2:]': word[-2:],
            'word.isupper()': word.isupper(),
            'word.istitle()': word.istitle(),
            'word.isdigit()': word.isdigit(),
            'postag': pos,
            'postag[:2]': pos[:2] if pos else '',
        }

        if i > 0:
            prev_word = '' if pd.isna(sent[i - 1].get('word', '')) else str(sent[i - 1].get('word', ''))
            prev_pos = '' if pd.isna(sent[i - 1].get('pos', '')) else str(sent[i - 1].get('pos', ''))
            feat.update({
                '-1:word.lower()': prev_word.lower(),
                '-1:word.istitle()': prev_word.istitle(),
                '-1:word.isupper()': prev_word.isupper(),
                '-1:postag': prev_pos,
                '-1:postag[:2]': prev_pos[:2] if prev_pos else '',
            })
        else:
            feat['BOS'] = True

        if i < len(sent) - 1:
            next_word = '' if pd.isna(sent[i + 1].get('word', '')) else str(sent[i + 1].get('word', ''))
            next_pos = '' if pd.isna(sent[i + 1].get('pos', '')) else str(sent[i + 1].get('pos', ''))
            feat.update({
                '+1:word.lower()': next_word.lower(),
                '+1:word.istitle()': next_word.istitle(),
                '+1:word.isupper()': next_word.isupper(),
                '+1:postag': next_pos,
                '+1:postag[:2]': next_pos[:2] if next_pos else '',
            })
        else:
            feat['EOS'] = True

        features.append(feat)
    return features

def sent2labels(sent):
    return [tok["ner"] for tok in sent]

X_train = [sent2features(s) for s in train_sentences]
y_train = [sent2labels(s) for s in train_sentences]

X_test = [sent2features(s) for s in test_sentences]
y_test = [sent2labels(s) for s in test_sentences]

In [34]:
crf = CRF(
    algorithm="lbfgs",
    c1=0.1,
    c2=0.1,
    max_iterations=100,
    all_possible_transitions=True
)

crf.fit(X_train, y_train)

,algorithm,'lbfgs'
,min_freq,None
,all_possible_states,None
,all_possible_transitions,True
,c1,0.1
,c2,0.1
,max_iterations,100
,num_memories,None
,epsilon,None
,period,None
,delta,None


In [35]:
y_pred = crf.predict(X_test)

flat_true = [label for sent in y_test for label in sent]
flat_pred = [label for sent in y_pred for label in sent]

print("\nToken accuracy:", accuracy_score(flat_true, flat_pred))
print("Flat F1 (micro):", flat_f1_score(y_test, y_pred, average="micro"))
print("Flat F1 (macro):", flat_f1_score(y_test, y_pred, average="macro"))

print("\nClassification report:")
print(flat_classification_report(y_test, y_pred))


Token accuracy: 0.8878504672897196
Flat F1 (micro): 0.8878504672897196
Flat F1 (macro): 0.5021873991052073

Classification report:
              precision    recall  f1-score   support

       B-LOC       0.60      0.75      0.67         4
      B-MISC       1.00      0.33      0.50         3
       B-ORG       0.33      0.50      0.40         4
       B-PER       0.50      0.17      0.25         6
       I-LOC       1.00      1.00      1.00         2
      I-MISC       0.00      0.00      0.00         1
       I-ORG       0.23      1.00      0.38         3
       I-PER       0.67      0.25      0.36         8
           O       0.97      0.96      0.96       183

    accuracy                           0.89       214
   macro avg       0.59      0.55      0.50       214
weighted avg       0.91      0.89      0.89       214



C:\Users\lars\Documents\School\group50-text-mining\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\lars\Documents\School\group50-text-mining\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\lars\Documents\School\group50-text-mining\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_p